# Day 2 Morning: 從 Naive RAG 到 Agentic RAG + Agentic RAG 實作

---
## 環境安裝

In [ ]:
%%capture
!pip install langchain langgraph chromadb langchain-community langchain-openai langchain-chroma

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY"

In [ ]:
# 基本 imports
from typing import Optional
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# 初始化 LLM
llm = ChatOpenAI(model="gpt-5.4-mini-2026-03-17", temperature=0)

---
# Part 1: 從 Naive RAG 到 Agentic RAG

## 1.1 傳統 RAG 的限制

傳統 RAG（Retrieval-Augmented Generation）的流程很簡單：**查詢 → 檢索 → 生成**。但這個流程有幾個關鍵限制：

| 限制 | 說明 |
|------|------|
| **單次檢索** | 只做一次檢索，無法根據結果品質決定是否需要重新檢索 |
| **無品質把關** | 檢索到的文件不一定跟問題相關，但系統照單全收 |
| **幻覺風險** | LLM 可能根據不相關的上下文「編造」答案，使用者無法察覺 |
| **無路由能力** | 所有問題都走同一條路徑，無法根據問題類型選擇最佳策略 |
| **無自我修正** | 如果回答品質差，系統不會自動重試或改進 |

這些限制在簡單問答場景下還可以接受，但在企業級應用中就顯得力不從心。

## 1.2 RAG 三代演進：Naive → Enhanced → Agentic

### Naive RAG（第一代）
```
Query → Retrieve → Generate → Answer
```
- 最簡單的 retrieve-then-read 模式
- 沒有任何品質檢查或反饋迴路

### Enhanced RAG（第二代）
```
Query → Rewrite → Retrieve → Re-rank → Generate → Answer
```
- 加入 **Query Rewriting**：改寫使用者查詢，提升檢索品質
- 加入 **Re-ranking**：對檢索結果重新排序，過濾雜訊
- 仍然是單向管線，沒有反饋機制

### Agentic RAG（第三代）
```
Query → Route → [Retrieve/Search/SQL] → Grade → Generate → Check → (retry?) → Answer
```
- **路由（Routing）**：根據問題類型選擇最佳資料來源
- **品質評分（Grading）**：自動評估檢索結果的相關性
- **幻覺檢測（Hallucination Check）**：驗證回答是否有根據
- **自我修正（Self-Correction）**：失敗時自動重試、改寫、換策略
- **迭代式（Iterative）**：可以多次循環直到得到高品質回答

## 1.3 Naive RAG 實作（對照組）

先建一個最基本的 Naive RAG 作為對照，後面會用 Agentic RAG 來改進它。

In [ ]:
# 準備範例文件（中文 AI/科技主題）
sample_docs = [
    Document(
        page_content="大型語言模型（LLM）是基於 Transformer 架構的深度學習模型，透過大量文本資料進行預訓練。"
        "GPT-4、Claude、Gemini 等都是知名的 LLM。這些模型具備強大的語言理解和生成能力，"
        "可以進行翻譯、摘要、問答、程式碼生成等多種任務。LLM 的參數量通常在數十億到數千億之間。",
        metadata={"source": "ai_basics", "topic": "LLM"}
    ),
    Document(
        page_content="RAG（Retrieval-Augmented Generation）是一種結合檢索與生成的技術框架。"
        "它的核心思想是：在生成回答之前，先從外部知識庫中檢索相關文件，然後將這些文件作為上下文提供給 LLM。"
        "RAG 可以有效減少 LLM 的幻覺問題，並讓模型能夠存取最新的、特定領域的知識。"
        "常見的 RAG 實作使用向量資料庫（如 Chroma、Pinecone）來儲存文件的嵌入向量。",
        metadata={"source": "rag_intro", "topic": "RAG"}
    ),
    Document(
        page_content="AI Agent（人工智慧代理）是能夠自主規劃、決策和執行任務的 AI 系統。"
        "與傳統的 AI 應用不同，Agent 具備以下特性：自主性（能獨立做出決策）、"
        "反應性（能感知環境變化並做出回應）、社會性（能與其他 Agent 或人類互動）。"
        "常見的 Agent 框架包括 LangGraph、AutoGen、CrewAI 等。"
        "Agent 可以使用各種工具（Tools），如搜尋引擎、資料庫查詢、API 呼叫等。",
        metadata={"source": "agent_intro", "topic": "Agent"}
    ),
    Document(
        page_content="向量資料庫是專門用來儲存和檢索高維向量的資料庫系統。"
        "在 AI 應用中，文本、圖片等資料會被轉換成嵌入向量（Embedding），然後儲存在向量資料庫中。"
        "檢索時，系統會計算查詢向量與儲存向量之間的相似度（如餘弦相似度），返回最相似的結果。"
        "常見的向量資料庫包括 Chroma、Pinecone、Weaviate、Milvus 等。"
        "向量資料庫是 RAG 系統的核心基礎設施之一。",
        metadata={"source": "vector_db", "topic": "VectorDB"}
    ),
    Document(
        page_content="Prompt Engineering（提示工程）是設計和優化 LLM 輸入提示的技術。"
        "好的 Prompt 可以顯著提升模型的回答品質。常見的技巧包括："
        "Few-shot Learning（提供範例）、Chain-of-Thought（鏈式思維推理）、"
        "Role Playing（角色扮演）等。Prompt Engineering 是使用 LLM 的基礎技能，"
        "但它無法解決所有問題，例如知識過時、幻覺等仍需要 RAG 等技術來輔助。",
        metadata={"source": "prompt_eng", "topic": "Prompt"}
    ),
]

print(f"準備了 {len(sample_docs)} 篇範例文件")

In [ ]:
# 文件切割 + 建立向量資料庫
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50,
)
splits = text_splitter.split_documents(sample_docs)
print(f"切割成 {len(splits)} 個 chunks")

# 建立 Chroma 向量資料庫
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name="naive_rag_demo",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("向量資料庫建立完成！")

In [ ]:
# Naive RAG：簡單的 retrieve + generate
naive_rag_prompt = ChatPromptTemplate.from_template(
    """根據以下參考文件回答問題。

參考文件：
{context}

問題：{question}

回答："""
)

def naive_rag(question: str) -> str:
    """最基本的 Naive RAG"""
    # 1. 檢索
    docs = retriever.invoke(question)
    context = "\n\n".join([d.page_content for d in docs])
    
    # 2. 生成
    chain = naive_rag_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    
    return answer, docs

In [ ]:
# 測試 Naive RAG — 正常情境
answer, docs = naive_rag("什麼是 RAG？")
print("問題：什麼是 RAG？")
print(f"\n檢索到 {len(docs)} 篇文件")
for i, doc in enumerate(docs):
    print(f"  文件 {i+1}: {doc.page_content[:60]}...")
print(f"\n回答：{answer}")

In [ ]:
# 測試 Naive RAG — 暴露問題
# 問題 1：知識庫中沒有的問題（會產生幻覺）
answer, docs = naive_rag("台灣 2025 年的 GDP 成長率是多少？")
print("問題：台灣 2025 年的 GDP 成長率是多少？")
print(f"檢索到的文件與問題完全無關，但 Naive RAG 照樣生成答案：")
print(f"回答：{answer}")
print("\n⚠️ 這就是 Naive RAG 的問題：沒有品質檢查，即使檢索結果不相關也會硬生成答案！")

In [ ]:
# 問題 2：模糊查詢，檢索效果差
answer, docs = naive_rag("那個東西怎麼用？")
print("問題：那個東西怎麼用？")
print(f"回答：{answer}")
print("\n⚠️ 模糊查詢讓檢索結果品質更差，Naive RAG 無法改寫查詢來改善這個問題！")

## 1.4 Agentic RAG 核心能力

Agentic RAG 用「代理」（Agent）的思維來解決上述問題，它具備四大核心能力：

### 1. 自我修正（Self-Correction）
- 當檢索結果品質不佳時，自動改寫查詢並重新檢索
- 當回答品質不達標時，自動重新生成
- 設有重試上限，避免無限循環

### 2. 智慧路由（Routing）
- 根據問題類型選擇最佳處理路徑
- 例如：事實查詢走向量資料庫、時事問題走網路搜尋、資料查詢走 SQL

### 3. 品質評分（Quality Grading）
- 用 LLM 評估每份檢索文件的相關性
- 過濾掉不相關的文件，只保留有用的上下文

### 4. 幻覺檢測（Hallucination Detection）
- 檢查回答是否有充分的文件支持
- 防止 LLM 「無中生有」

## 1.5 Self-RAG / Corrective RAG 概念解析

### Self-RAG
Self-RAG 的核心概念是讓 LLM **自己判斷**是否需要檢索、檢索結果是否有用、回答是否可靠：

```
問題輸入
  ↓
需要檢索嗎？ → 否 → 直接回答
  ↓ 是
檢索文件
  ↓
文件相關嗎？ → 否 → 改寫查詢，重新檢索
  ↓ 是
生成回答
  ↓
回答有根據嗎？ → 否 → 重新生成
  ↓ 是
輸出回答
```

### Corrective RAG (CRAG)
CRAG 特別強調對檢索結果的「修正」：
- **正確**：文件高度相關 → 直接使用
- **模糊**：文件部分相關 → 擷取相關段落
- **錯誤**：文件不相關 → 丟棄，改用網路搜尋作為備援

這兩個概念是 Agentic RAG 的理論基礎。

## 1.6 LangGraph 在 Agentic RAG 中的角色

為什麼用 **Graph 圖架構** 來實作 Agentic RAG？

| 特性 | Chain（鏈式） | Graph（圖式） |
|------|-------------|-------------|
| 流程控制 | 線性，固定順序 | 非線性，條件分支 |
| 循環 | 不支持 | 原生支持 |
| 條件路由 | 困難 | 簡單直覺 |
| 狀態管理 | 手動傳遞 | 內建 State |
| 除錯 | 困難 | 視覺化圖形 |

LangGraph 的優勢：
1. **條件邊（Conditional Edge）**：根據評分結果決定下一步去向
2. **循環（Cycle）**：支持 retry 邏輯（重新檢索、重新生成）
3. **狀態共享（Shared State）**：所有節點都能讀寫同一個 State
4. **可視化（Visualization）**：編譯後可直接看到完整流程圖

---
# Part 2: Agentic RAG 實作

## 2.1 系統架構規劃

完整的 Agentic RAG 架構如下：

```
                    ┌──────────────┐
                    │   Question   │
                    └──────┬───────┘
                           │
                    ┌──────▼───────┐
                    │   Router     │
                    └──┬───┬───┬───┘
                       │   │   │
          ┌────────────┘   │   └────────────┐
          │                │                │
   ┌──────▼──────┐ ┌──────▼──────┐ ┌───────▼──────┐
   │ VectorStore │ │ Web Search  │ │Direct Answer │
   └──────┬──────┘ └──────┬──────┘ └───────┬──────┘
          │                │                │
          └────────┬───────┘                │
                   │                        │
            ┌──────▼───────┐                │
            │   Grader     │                │
            └──┬───────┬───┘                │
               │       │                    │
          足夠  │       │ 不足               │
               │       │                    │
               │  ┌────▼─────┐              │
               │  │ Rewrite  │              │
               │  └────┬─────┘              │
               │       │ (重新檢索)          │
               │       └──→ VectorStore     │
               │                            │
        ┌──────▼───────┐                    │
        │  Generator   │◄───────────────────┘
        └──────┬───────┘
               │
        ┌──────▼───────────┐
        │Hallucination     │
        │   Checker        │
        └──┬───────────┬───┘
           │           │
      通過  │           │ 未通過
           │           │
    ┌──────▼──┐   ┌────▼──────┐
    │  END    │   │ Retry /   │
    │(輸出答案)│   │ Fallback  │
    └─────────┘   └───────────┘
```

In [ ]:
# 定義完整的 Agentic RAG 狀態
class AgenticRAGState(TypedDict):
    question: str                           # 使用者原始問題
    rewritten_query: Optional[str]          # 改寫後的查詢
    route: Optional[str]                    # 路由決策: "vectorstore", "web_search", "direct_answer"
    documents: list                         # 檢索到的原始文件
    filtered_documents: list                # 經過評分過濾後的文件
    answer: Optional[str]                   # 生成的回答
    hallucination_check: Optional[bool]     # 幻覺檢測結果
    quality_score: Optional[float]          # 回答品質評分
    retry_count: int                        # 重試次數
    needs_escalation: bool                  # 是否需要轉介人工

print("State 定義完成！")
print("欄位：", list(AgenticRAGState.__annotations__.keys()))

## 2.2 Router 路由節點

使用 LLM + Structured Output 來分類問題，決定走哪條路徑。

In [ ]:
from pydantic import BaseModel, Field

# 定義路由的結構化輸出
class RouteDecision(BaseModel):
    """路由決策結果"""
    route: str = Field(
        description="路由決策，必須是以下其中之一：'vectorstore'（知識庫查詢）、'web_search'（網路搜尋）、'direct_answer'（直接回答）"
    )
    reason: str = Field(
        description="選擇此路由的原因"
    )

# 建立帶有結構化輸出的路由 LLM
router_llm = llm.with_structured_output(RouteDecision)

router_prompt = ChatPromptTemplate.from_template(
    """你是一個問題路由器。根據使用者的問題，決定最佳的處理路徑。

路由選項：
- vectorstore：問題與 AI、LLM、RAG、Agent、向量資料庫、Prompt Engineering 等技術主題相關
- web_search：問題涉及即時資訊、新聞、特定數據、或知識庫可能沒有的內容
- direct_answer：問題是簡單的常識問題、打招呼、或不需要外部資料就能回答的問題

使用者問題：{question}"""
)

def route_question(state: AgenticRAGState) -> AgenticRAGState:
    """路由節點：決定問題的處理路徑"""
    print("[Router] 分析問題路由...")
    chain = router_prompt | router_llm
    result = chain.invoke({"question": state["question"]})
    print(f"[Router] 決策：{result.route}（原因：{result.reason}）")
    return {**state, "route": result.route}

In [ ]:
# 測試路由
test_questions = [
    "什麼是 RAG？",                    # 應該走 vectorstore
    "今天台北天氣如何？",                # 應該走 web_search
    "你好，請問你是誰？",                # 應該走 direct_answer
    "LangGraph 怎麼實作 Agent？",       # 應該走 vectorstore
    "台積電今天的股價是多少？",           # 應該走 web_search
]

for q in test_questions:
    test_state = AgenticRAGState(
        question=q, rewritten_query=None, route=None,
        documents=[], filtered_documents=[], answer=None,
        hallucination_check=None, quality_score=None,
        retry_count=0, needs_escalation=False,
    )
    result = route_question(test_state)
    print(f"  → {q} → {result['route']}")
    print()

## 2.3 Rewrite Query 查詢改寫節點

當使用者問題太模糊或檢索效果不佳時，用 LLM 改寫查詢。

In [ ]:
rewrite_prompt = ChatPromptTemplate.from_template(
    """你是一個查詢改寫專家。將使用者的問題改寫成更適合向量資料庫檢索的版本。

改寫原則：
1. 讓查詢更具體、更明確
2. 補充可能的關鍵詞
3. 去除口語化表達
4. 保持原始意圖不變

原始問題：{question}

改寫後的查詢（只輸出改寫結果，不要其他說明）："""
)

def rewrite_query(state: AgenticRAGState) -> AgenticRAGState:
    """查詢改寫節點"""
    print("[Rewrite] 改寫查詢中...")
    chain = rewrite_prompt | llm | StrOutputParser()
    rewritten = chain.invoke({"question": state["question"]})
    print(f"[Rewrite] 原始：{state['question']}")
    print(f"[Rewrite] 改寫：{rewritten}")
    return {**state, "rewritten_query": rewritten}

In [ ]:
# 測試查詢改寫
test_vague_questions = [
    "那個東西怎麼用？",
    "AI 跟以前有什麼不一樣",
    "怎麼讓模型不要亂講話",
]

for q in test_vague_questions:
    test_state = AgenticRAGState(
        question=q, rewritten_query=None, route=None,
        documents=[], filtered_documents=[], answer=None,
        hallucination_check=None, quality_score=None,
        retry_count=0, needs_escalation=False,
    )
    result = rewrite_query(test_state)
    print()

## 2.4 Retrieval 知識檢索節點

從向量資料庫檢索相關文件。如果有改寫查詢，優先使用改寫後的版本。

In [ ]:
def retrieve_documents(state: AgenticRAGState) -> AgenticRAGState:
    """檢索節點：從向量資料庫檢索文件"""
    # 如果有改寫過的查詢就用改寫版，否則用原始問題
    query = state["rewritten_query"] or state["question"]
    print(f"[Retrieve] 使用查詢：{query}")

    docs = retriever.invoke(query)
    print(f"[Retrieve] 檢索到 {len(docs)} 篇文件")
    for i, doc in enumerate(docs):
        print(f"  文件 {i+1}: {doc.page_content[:80]}...")

    return {**state, "documents": docs}

In [ ]:
# 模擬網路搜尋（實際應用中可接入 Tavily、SerpAPI 等）
def web_search(state: AgenticRAGState) -> AgenticRAGState:
    """網路搜尋節點（模擬）"""
    query = state["rewritten_query"] or state["question"]
    print(f"[WebSearch] 搜尋：{query}")

    # 模擬搜尋結果
    mock_results = [
        Document(
            page_content=f"[網路搜尋結果] 關於「{query}」的最新資訊：這是一個模擬的搜尋結果。"
            f"在實際應用中，這裡會接入真實的搜尋 API（如 Tavily Search）來取得即時網路資訊。",
            metadata={"source": "web_search", "url": "https://example.com"}
        ),
    ]
    print(f"[WebSearch] 取得 {len(mock_results)} 筆搜尋結果")

    return {**state, "documents": mock_results, "filtered_documents": mock_results}

## 2.5 Grader 品質評分節點

用 LLM 逐一評估每份檢索文件的相關性，過濾掉無關的文件。

In [ ]:
# 定義評分結構化輸出
class GradeResult(BaseModel):
    """文件相關性評分"""
    is_relevant: bool = Field(description="此文件是否與問題相關")
    reason: str = Field(description="判斷原因")

grader_llm = llm.with_structured_output(GradeResult)

grader_prompt = ChatPromptTemplate.from_template(
    """你是一個文件相關性評分專家。請判斷以下文件是否與使用者的問題相關。

評分標準：
- 如果文件包含與問題直接相關的資訊，判定為「相關」
- 如果文件與問題主題無關或只有微弱關聯，判定為「不相關」

文件內容：
{document}

使用者問題：{question}"""
)

def grade_documents(state: AgenticRAGState) -> AgenticRAGState:
    """品質評分節點：過濾不相關的文件"""
    print("[Grader] 評估文件相關性...")
    chain = grader_prompt | grader_llm

    filtered = []
    for i, doc in enumerate(state["documents"]):
        result = chain.invoke({
            "document": doc.page_content,
            "question": state["question"]
        })
        status = "相關 ✓" if result.is_relevant else "不相關 ✗"
        print(f"  文件 {i+1}: {status}（{result.reason}）")
        if result.is_relevant:
            filtered.append(doc)

    print(f"[Grader] 通過評分：{len(filtered)}/{len(state['documents'])} 篇")
    return {**state, "filtered_documents": filtered}

In [ ]:
# 測試 Grader
test_state = AgenticRAGState(
    question="什麼是 RAG？", rewritten_query=None, route=None,
    documents=[], filtered_documents=[], answer=None,
    hallucination_check=None, quality_score=None,
    retry_count=0, needs_escalation=False,
)
test_state = retrieve_documents(test_state)
print()
test_state = grade_documents(test_state)

## 2.6 Generator 回答生成節點

根據過濾後的文件，生成附帶引用來源的回答。

In [ ]:
generator_prompt = ChatPromptTemplate.from_template(
    """你是一個專業的 AI 助教。根據以下參考文件回答使用者的問題。

        重要規則：
        1. 只根據提供的參考文件來回答，不要捏造資訊
        2. 如果參考文件不足以回答問題，請明確說明
        3. 在回答中標注引用來源
        
        參考文件：
        {context}
        
        使用者問題：{question}
        
        回答："""
)

def generate_answer(state: AgenticRAGState) -> AgenticRAGState:
    """回答生成節點"""
    print("[Generator] 生成回答中...")

    # 使用過濾後的文件
    docs = state["filtered_documents"]
    if not docs:
        print("[Generator] 沒有相關文件，生成備用回答")
        return {**state, "answer": "很抱歉，我在知識庫中找不到與您問題相關的資訊。建議改用網路搜尋或換一種方式提問。"}

    # 組合上下文，附帶來源資訊
    context_parts = []
    for i, doc in enumerate(docs):
        source = doc.metadata.get("source", "unknown")
        context_parts.append(f"[來源: {source}] {doc.page_content}")
    context = "\n\n".join(context_parts)

    chain = generator_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": state["question"]})
    print(f"[Generator] 回答已生成（{len(answer)} 字）")

    return {**state, "answer": answer}

## 2.7 Hallucination Checker 幻覺檢測節點

檢查生成的回答是否真的有文件支持，防止模型「無中生有」。

In [ ]:
class HallucinationResult(BaseModel):
    """幻覺檢測結果"""
    is_grounded: bool = Field(description="回答是否有充分的文件支持")
    reason: str = Field(description="判斷原因")

hallucination_llm = llm.with_structured_output(HallucinationResult)

hallucination_prompt = ChatPromptTemplate.from_template(
    """你是一個幻覺檢測專家。請判斷以下回答是否有充分的文件支持。

        判斷標準：
        - 回答中的每個事實性陳述都應該能在參考文件中找到依據
        - 如果回答包含參考文件中沒有的資訊（尤其是具體數字、日期、事件），判定為「幻覺」
        - 合理的推論和總結是允許的
        
        參考文件：
        {documents}
        
        生成的回答：
        {answer}
        
        此回答是否有充分的文件支持？"""
)

def check_hallucination(state: AgenticRAGState) -> AgenticRAGState:
    """幻覺檢測節點"""
    print("[HallucinationCheck] 檢測回答可靠性...")

    # 如果沒有文件（direct_answer 路徑），跳過檢查
    if not state["filtered_documents"]:
        print("[HallucinationCheck] 無參考文件，跳過檢查")
        return {**state, "hallucination_check": True}

    docs_text = "\n\n".join([d.page_content for d in state["filtered_documents"]])
    chain = hallucination_prompt | hallucination_llm
    result = chain.invoke({
        "documents": docs_text,
        "answer": state["answer"]
    })

    status = "有根據 ✓" if result.is_grounded else "疑似幻覺 ✗"
    print(f"[HallucinationCheck] {status}（{result.reason}）")

    return {**state, "hallucination_check": result.is_grounded}

## 2.8 Fallback / Escalate 降級轉介節點

當重試次數超過上限時，生成降級回答或標記需要人工處理。

In [ ]:
MAX_RETRIES = 2  # 最大重試次數

def fallback_response(state: AgenticRAGState) -> AgenticRAGState:
    """降級轉介節點"""
    print("[Fallback] 觸發降級機制...")
    fallback_answer = (
        f"很抱歉，我嘗試了多次但無法為您的問題「{state['question']}」提供可靠的回答。\n"
        f"建議您：\n"
        f"1. 嘗試用不同的方式重新提問\n"
        f"2. 將問題拆解成更小的子問題\n"
        f"3. 聯繫人工客服取得協助"
    )
    return {
        **state,
        "answer": fallback_answer,
        "needs_escalation": True,
    }

def direct_answer(state: AgenticRAGState) -> AgenticRAGState:
    """直接回答節點（不需要檢索）"""
    print("[DirectAnswer] 直接生成回答...")
    chain = ChatPromptTemplate.from_template(
        "請直接回答以下問題：{question}"
    ) | llm | StrOutputParser()
    answer = chain.invoke({"question": state["question"]})
    return {
        **state,
        "answer": answer,
        "hallucination_check": True,  # 直接回答不做幻覺檢查
    }

## 2.9 組裝完整 Agentic RAG 系統

使用 LangGraph 把所有節點串接起來，加上條件邊實現動態路由。

In [ ]:
from langgraph.graph import StateGraph, START, END

# ===== 條件判斷函數 =====

def route_after_router(state: AgenticRAGState) -> str:
    """Router 之後的路由邏輯"""
    route = state["route"]
    if route == "vectorstore":
        return "retrieve"
    elif route == "web_search":
        return "web_search"
    else:
        return "direct_answer"

def route_after_grader(state: AgenticRAGState) -> str:
    """Grader 之後的路由邏輯"""
    # 如果有足夠的相關文件 → 生成回答
    if len(state["filtered_documents"]) > 0:
        return "generate"
    # 沒有足夠的相關文件 → 查看是否還能重試
    if state["retry_count"] < MAX_RETRIES:
        return "rewrite"
    return "fallback"

def route_after_hallucination(state: AgenticRAGState) -> str:
    """幻覺檢測之後的路由邏輯"""
    if state["hallucination_check"]:
        return "end"
    # 幻覺被檢測到 → 重試或降級
    if state["retry_count"] < MAX_RETRIES:
        return "regenerate"
    return "fallback"

# ===== 包裝函數（處理 retry_count 遞增）=====

def rewrite_and_increment(state: AgenticRAGState) -> AgenticRAGState:
    """改寫查詢並遞增重試次數"""
    new_state = rewrite_query(state)
    new_state["retry_count"] = state["retry_count"] + 1
    return new_state

def regenerate_answer(state: AgenticRAGState) -> AgenticRAGState:
    """重新生成回答（幻覺檢測失敗時）"""
    print("[Regenerate] 幻覺檢測未通過，重新生成回答...")
    new_state = {**state, "retry_count": state["retry_count"] + 1}
    return generate_answer(new_state)

In [ ]:
# ===== 建構 Graph =====

workflow = StateGraph(AgenticRAGState)

# 添加所有節點
workflow.add_node("router", route_question)
workflow.add_node("retrieve", retrieve_documents)
workflow.add_node("web_search", web_search)
workflow.add_node("direct_answer", direct_answer)
workflow.add_node("grader", grade_documents)
workflow.add_node("rewrite", rewrite_and_increment)
workflow.add_node("generate", generate_answer)
workflow.add_node("hallucination_check", check_hallucination)
workflow.add_node("regenerate", regenerate_answer)
workflow.add_node("fallback", fallback_response)

# ===== 連接邊 =====

# 入口 → 路由
workflow.add_edge(START, "router")

# 路由 → 三條路徑
workflow.add_conditional_edges(
    "router",
    route_after_router,
    {
        "retrieve": "retrieve",
        "web_search": "web_search",
        "direct_answer": "direct_answer",
    }
)

# 檢索 → 評分
workflow.add_edge("retrieve", "grader")

# 評分 → 生成 / 改寫 / 降級
workflow.add_conditional_edges(
    "grader",
    route_after_grader,
    {
        "generate": "generate",
        "rewrite": "rewrite",
        "fallback": "fallback",
    }
)

# 改寫 → 重新檢索
workflow.add_edge("rewrite", "retrieve")

# 網路搜尋 → 直接生成（不經過 grader）
workflow.add_edge("web_search", "generate")

# 生成 → 幻覺檢測
workflow.add_edge("generate", "hallucination_check")

# 幻覺檢測 → 結束 / 重新生成 / 降級
workflow.add_conditional_edges(
    "hallucination_check",
    route_after_hallucination,
    {
        "end": END,
        "regenerate": "regenerate",
        "fallback": "fallback",
    }
)

# 重新生成 → 再次幻覺檢測
workflow.add_edge("regenerate", "hallucination_check")

# 直接回答 → 結束
workflow.add_edge("direct_answer", END)

# 降級 → 結束
workflow.add_edge("fallback", END)

# 編譯
app = workflow.compile()
print("Agentic RAG 系統編譯完成！")

In [ ]:
# 視覺化完整流程圖
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception:
    # 如果無法渲染圖片，顯示 Mermaid 文字
    print(app.get_graph().draw_mermaid())

In [ ]:
# 定義一個方便的執行函數
def run_agentic_rag(question: str) -> dict:
    """執行 Agentic RAG"""
    initial_state = AgenticRAGState(
        question=question,
        rewritten_query=None,
        route=None,
        documents=[],
        filtered_documents=[],
        answer=None,
        hallucination_check=None,
        quality_score=None,
        retry_count=0,
        needs_escalation=False,
    )

    print("=" * 60)
    print(f"問題：{question}")
    print("=" * 60)

    result = app.invoke(initial_state)

    print("\n" + "=" * 60)
    print("最終結果：")
    print(f"  路由：{result['route']}")
    print(f"  重試次數：{result['retry_count']}")
    print(f"  幻覺檢測：{'通過' if result['hallucination_check'] else '未通過'}")
    print(f"  需要轉介：{'是' if result['needs_escalation'] else '否'}")
    print(f"  回答：\n{result['answer']}")
    print("=" * 60)

    return result

In [ ]:
# 測試 1：知識庫問題 → vectorstore 路徑
result1 = run_agentic_rag("什麼是 RAG？它如何減少 LLM 的幻覺問題？")

In [ ]:
# 測試 2：即時資訊 → web_search 路徑
result2 = run_agentic_rag("今天台北的天氣如何？")

In [ ]:
# 測試 3：簡單問題 → direct_answer 路徑
result3 = run_agentic_rag("你好！請自我介紹一下。")

In [ ]:
# 測試 4：知識庫中沒有的問題 → 可能觸發 rewrite 或 fallback
result4 = run_agentic_rag("台灣 2025 年的 GDP 成長率是多少？")

In [ ]:
# 測試 5：模糊查詢 → 改寫後效果應更好
result5 = run_agentic_rag("那個 Agent 框架要怎麼用啊？")

### Naive RAG vs Agentic RAG 比較

| 面向 | Naive RAG | Agentic RAG |
|------|-----------|-------------|
| 路由 | 所有問題走同一路徑 | 根據問題類型智慧路由 |
| 檢索品質 | 照單全收 | LLM 評分過濾不相關文件 |
| 查詢改寫 | 無 | 模糊查詢自動改寫 |
| 幻覺檢測 | 無 | 自動檢測並重新生成 |
| 容錯 | 無 | 重試 + 降級機制 |
| 適用場景 | 簡單問答 | 企業級應用 |

---
# Part 3: MCP（Model Context Protocol）

MCP 是 Anthropic 提出的標準化 AI 工具協定，讓任何 Agent 都能透過統一介面存取工具。

| 問題 | MCP 解法 |
|------|----------|
| 每個框架有自己的 Tool 格式 | 統一協議，一次實作處處可用 |
| 工具散落各處難以共用 | 封裝成獨立 Server，多 Agent 共用 |
| 新增工具需改 Agent 程式碼 | Server 端更新，Client 自動發現 |

> **類比**：MCP 就像 USB 介面——出現之前每種設備各自接口不相容；有了 MCP，所有 AI 工具都能即插即用。

In [ ]:
%%capture
!pip install fastmcp langchain-mcp-adapters

---
## 3.1 建立 MCP Server（hp_mcp_server.py）

將 HP 的三個業務工具封裝成一個 MCP Server：
- `check_inventory` — 查詢庫存
- `check_warranty` — 確認保固狀態
- `get_repair_sop` — 取得維修 SOP

In [ ]:
# 將 MCP Server 寫入檔案（與 scripts/hp_mcp_server.py 內容相同）
# Colab 環境下先寫到本機，再由 MCP Client 以 subprocess 啟動
server_code = """
from fastmcp import FastMCP

mcp = FastMCP("HP Tools")

# ─── 模擬資料庫 ──────────────────────────────────────────────────
_INVENTORY = {
    "HP EliteBook 840": {"stock": 15, "location": "台北倉"},
    "HP LaserJet Pro":  {"stock": 3,  "location": "新竹倉"},
    "HP ZBook Studio":  {"stock": 0,  "location": "缺貨"},
}

_WARRANTY = {
    "SN12345": {"status": "有效", "expire": "2026-12-31", "model": "HP EliteBook 840"},
    "SN99999": {"status": "過期", "expire": "2024-01-01", "model": "HP LaserJet Pro"},
}

_SOP = {
    "無法開機":   "1. 確認電源線 2. 嘗試硬重啟 3. 進 BIOS 診斷 4. 聯絡技術支援",
    "印表機卡紙": "1. 關閉電源 2. 打開前蓋 3. 輕輕取出紙張 4. 重新對齊紙張後重啟",
}

# ─── MCP Tools ───────────────────────────────────────────────────
@mcp.tool()
def check_inventory(product: str) -> dict:
    \"\"\"查詢 HP 產品庫存數量與倉庫位置\"\"\"
    return _INVENTORY.get(product, {"stock": "未知", "location": "查無此產品"})

@mcp.tool()
def check_warranty(serial_number: str) -> dict:
    \"\"\"根據序號確認 HP 產品保固狀態與到期日\"\"\"
    return _WARRANTY.get(serial_number, {"status": "查無資料", "expire": None})

@mcp.tool()
def get_repair_sop(issue: str) -> str:
    \"\"\"取得 HP 常見問題的維修 SOP 步驟\"\"\"
    for key, sop in _SOP.items():
        if key in issue:
            return sop
    return "請聯絡 HP 技術支援：0800-012-345"

# ─── MCP Resources ───────────────────────────────────────────────
PRODUCT_CATALOG = [
    {"line": "EliteBook", "segment": "商務筆電", "range": "800 / 1000 系列"},
    {"line": "ZBook",     "segment": "工作站筆電", "range": "Studio / Fury"},
    {"line": "LaserJet",  "segment": "雷射印表機", "range": "Pro / Enterprise"},
]

@mcp.resource("kb://products")
def list_products() -> list:
    \"\"\"列出 HP 知識庫中的產品線（靜態目錄）\"\"\"
    return PRODUCT_CATALOG

if __name__ == "__main__":
    mcp.run(transport="stdio")
"""

with open("hp_mcp_server.py", "w") as f:
    f.write(server_code)

print("✅ hp_mcp_server.py 已建立（與 scripts/hp_mcp_server.py 內容相同）")


---
## 3.2 MCP Client 串接 LangGraph

`langchain-mcp-adapters` 提供 `MultiServerMCPClient`，讓 LangGraph Agent 自動發現並載入 MCP Server 上的工具。

```
LangGraph Agent
     │
     └─ MultiServerMCPClient
          └─ stdio → hp_mcp_server.py
                       ├─ check_inventory
                       ├─ check_warranty
                       └─ get_repair_sop
```

In [ ]:
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)

mcp_config = {
    "hp-tools": {
        "command": "python",
        "args": ["hp_mcp_server.py"],
        "transport": "stdio",
    }
}

async def run_mcp_agent(query: str):
    async with MultiServerMCPClient(mcp_config) as client:
        tools = client.get_tools()
        print(f"📦 從 MCP Server 載入 {len(tools)} 個工具：{[t.name for t in tools]}")

        agent = create_react_agent(llm, tools)
        result = await agent.ainvoke({"messages": [("user", query)]})

        for msg in result["messages"]:
            role = msg.__class__.__name__
            if hasattr(msg, "content") and msg.content:
                print(f"  [{role}] {msg.content[:300]}")
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  [Tool Call] {tc['name']}({tc['args']})")

print("=== 測試 1：查詢庫存 ===")
await run_mcp_agent("HP EliteBook 840 還有庫存嗎？在哪個倉庫？")


In [ ]:
# 測試 2：確認保固
print("=== 測試 2：確認保固 ===")
await run_mcp_agent("我的序號是 SN12345，請確認保固是否還有效？")

In [ ]:
# 測試 3：複合查詢（同時用到多個工具）
print("=== 測試 3：複合查詢 ===")
await run_mcp_agent("我的序號 SN99999 印表機卡紙了，保固還有效嗎？該怎麼修？")

---
## 3.3 MCP Resources — 靜態知識的標準存取方式

除了 `@mcp.tool()` 呼叫動態工具，MCP 還支援 `@mcp.resource()` 提供靜態資料（如產品目錄）。

| 類型 | 用途 | 觸發方式 |
|------|------|----------|
| `tool` | 動態操作（查詢、計算） | Agent 主動呼叫 |
| `resource` | 靜態資料（目錄、規格） | URI 存取 `kb://products` |
| `prompt` | 預設 Prompt 範本 | Client 請求 |

In [ ]:
# 示範 Resource 的寫法（概念說明，不需執行 Server）
resource_example = """
from fastmcp import FastMCP

mcp = FastMCP("HP 知識庫")

PRODUCT_CATALOG = [
    {"line": "EliteBook", "segment": "商務筆電", "range": "800 / 1000 系列"},
    {"line": "ZBook",     "segment": "工作站筆電", "range": "Studio / Fury"},
    {"line": "LaserJet",  "segment": "雷射印表機", "range": "Pro / Enterprise"},
]

# Resource：靜態產品目錄（URI 存取）
@mcp.resource("kb://products")
def list_products() -> list:
    return PRODUCT_CATALOG

# Tool：動態向量搜尋
@mcp.tool()
def search_kb(query: str, top_k: int = 5) -> list[str]:
    return [f"[模擬結果] 與 '{query}' 相關的文件內容..."] * top_k
"""

print(resource_example)
print("💡 重點：同一個 MCP Server 可同時服務多個 Agent（Agentic RAG + Deep Research）")


---
## 3.4 企業部署架構

HP 場景下的 MCP 部署方式，工具更新只需改 Server，所有 Agent 自動獲得最新版本：

```
┌─────────────────────────────────────────┐
│              LangGraph Agents           │
│  ┌──────────┐  ┌──────────┐  ┌────────┐ │
│  │ RAG Agent│  │Voice Agt │  │Research│ │
│  └────┬─────┘  └────┬─────┘  └───┬────┘ │
└───────┼─────────────┼─────────────┼──────┘
        │  MCP Client │             │
┌───────▼─────────────▼─────────────▼──────┐
│              MCP Servers                  │
│  ┌──────────┐  ┌──────────┐  ┌─────────┐ │
│  │庫存 Server│  │保固 Server│  │SOP Server│ │
│  └──────────┘  └──────────┘  └─────────┘ │
└───────────────────────────────────────────┘
```

In [ ]:
# Claude Desktop 設定方式（供參考）
import json

claude_config = {
    "mcpServers": {
        "hp-inventory": {
            "command": "python",
            "args": ["hp_mcp_server.py"]
        },
        "hp-knowledge": {
            "command": "python",
            "args": ["hp_kb_server.py"]
        }
    }
}

print("claude_desktop_config.json:")
print(json.dumps(claude_config, indent=2, ensure_ascii=False))
print()
print("✅ 將此設定放入 ~/Library/Application Support/Claude/claude_desktop_config.json")
print("   Claude Desktop 重啟後即可看到 HP Tools")